In [1]:
import os, yaml, sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
from general_utils.utils import decode_matlab_strings

In [51]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    monkey_name: str = 'baby1' # 'paul'
    date: str = '220226to527' # '230204' 
    img_size = 384
    folder_name = 'manyOO'  #  'fewer_occlusion'
    batch_size = 10
cfg = Cfg()

# TODO:
- try on different datasets
- substitute it on the previous scripts (loops in image_processing.computational_models)
- scale it on the cluster and extract feats

In [52]:

allimgs_path = f"{paths['livingstone_lab']}/tiziano/data/{cfg.monkey_name}_allimages{cfg.date}.mat"
with h5py.File(allimgs_path, "r") as f:
    try:
        refs = f["allimages"][:]      # shape (N, 1) of object refs
    except KeyError:
        refs = f["uniqueImage"][:]
    # end try:
    monkey_presentation_order = decode_matlab_strings(f, refs) # not sure if this stays the same in datasets with uniqueImage ... # ADD check
    monkey_presentation_order = sorted(set(monkey_presentation_order))


In [ ]:
from general_utils.utils import get_relevant_output_layers
from image_processing.utils import get_usual_transform
from torch.utils.data import DataLoader
import argparse
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = get_usual_transform(resize_size=cfg.img_size, normalize=False)
# task_list = get_relevant_output_layers(args.model_name, pkg=args.pkg)

dataset = ImageFolder(
    root=f"{paths['livingstone_lab']}/Stimuli/{cfg.folder_name}/",
    transform=transform,
    is_valid_file=lambda x: not x.endswith("Thumbs.db"), 
    allow_empty=True, 
)
dataloader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False)


In [ ]:
from general_utils.utils import print_wise
from torchvision.models.feature_extraction import create_feature_extractor
from image_processing.computational_models import img_dataloader_feature_extraction_loop

def img_feats_extraction_pooling(paths, rank, layer_name, model_name, model, dataloader, dataset_name, img_size, pooling, device):
    feats_save_name = f"{paths['livingstone_lab']}/tiziano/models/{dataset_name}_{model_name}_{img_size}_{layer_name}_features_{pooling}pool.npz"
    if os.path.exists(feats_save_name):
        print_wise(f"feature already computed at {feats_save_name}")
    else:
        feature_extractor = create_feature_extractor(model, return_nodes=[layer_name]).to(device)
        all_feats = img_dataloader_feature_extraction_loop(rank, feature_extractor, layer_name, dataloader, device, pooling=pooling)
        np.savez_compressed(feats_save_name, all_feats.T)
        print_wise(f"saved features at {feats_save_name}", rank=rank)
# EOF


In [ ]:
dl_ord = []
for i, (path, label) in enumerate(dataset.samples):
    print(i, path, label)
    dl_ord.append(os.path.basename(path))
    

0 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_01.jpg 0
1 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_02.jpg 0
2 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_03.jpg 0
3 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_04.jpg 0
4 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_05.jpg 0
5 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_06.jpg 0
6 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_07.jpg 0
7 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_08.jpg 0
8 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_09.jpg 0
9 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_10.jpg 0
10 /Users/tizianocausin/livingstone_lab_local/Stimuli/manyOO/BigAnimate/BigAnimate_11.jpg 

In [57]:
from image_processing.computational_models import map_image_order_from_ann_to_monkey
idx = map_image_order_from_ann_to_monkey(paths, cfg.monkey_name, cfg.date, base_dataset)

In [62]:
monkey_presentation_order

['-AUTOIMAGES-RU4418lg.jpg',
 '0-teak_tf53er.jpg',
 '0.jpg',
 '0000001604_Candyfloss_Collar[fullscreen].jpg',
 '0000004777_DSC_0894[fullscreen].jpg',
 '00000102404-SonyDCRTRV280Digital8camcorder-large.jpg',
 '00000103278-BowFlexXtreme2-large.jpg',
 '00000104285-HaroBikesMirra540Air-large.jpg',
 '00000109862-AmanaToasterOvenMicrowaveAMC5108AA-large.jpg',
 '00000135_LARGE.jpg',
 '000007103.jpg',
 '0001.jpg',
 '0002638858057_500X500.jpg',
 '0005465109884_215X215.jpg',
 '0008390000031_LG.jpg',
 '0036head.jpg',
 '0051507b.jpg',
 '0053_minox_binoculars.jpg',
 '009.-trashcan_lrg.jpg',
 '01.jpg',
 '010604_FaeroeIslandsP16a-10Kronur-(19)74-donated_f.jpg',
 '013180.jpg',
 '01393.fpx.jpg',
 '01dccef1-d4b2-4d2e-a317-f2ed46b70b4b_400.jpg',
 '020f7ec6-df08-40fa-b3a5-0dcfb762f085_4.jpg',
 '0230.jpg',
 '02565 ride on train.jpg',
 '029280e.jpg',
 '03015528000.jpg',
 '03070a.jpg',
 '0334.jpg',
 '0354_sportsocks_large.jpg',
 '040115_gadgets_razor_hmed10a.hmedium[1].jpg',
 '043033538959.jpg',
 '0430335389

In [65]:
act = []
for i in idx:
    act.append(os.path.basename(base_dataset.samples[i][0]))
    # print((base_dataset.samples[i]))

In [66]:
act == monkey_presentation_order

True

In [27]:
dataset = dataloader.dataset

for batch_idx, (images, labels) in enumerate(dataloader):
    start = batch_idx * dataloader.batch_size
    end = start + len(labels)

    batch_indices = dataset.indices[start:end]
    filenames = [
        os.path.basename(dataset.imagefolder.samples[i][0])
        for i in batch_indices
    ]

    print(f"batch {batch_idx}")
    print("files:", filenames)
    # ADD fix presentation order

batch 0
files: ['0-teak_tf53er.jpg', '07Model17RedSkiHelmet.jpg', '1054_detail.jpg', '1259_2341_popup1.jpg', '16234104.thl.jpg', '21SS8DBWDQL._SS500_.jpg', '26369935.thl.jpg', '26373229.thl.jpg', '26373355.thl.jpg', '26379142.thl.jpg']
batch 1
files: ['26381224.thl.jpg', '26387944.thl.jpg', '2sm.jpg', '31H95JKXWNL._SS500_.jpg', '35811374.thl.jpg', '54619901.jpg', '568eur6.jpg', '8027495.thl-1.jpg', '91180.jpg', 'ABEARMASK.jpg']
batch 2
files: ['AGARBACA8.jpg', 'AGARBCAN4.jpg', 'AJACKOL24.jpg', 'ALEI10.jpg', 'ALEI5.jpg', 'ALEI6.jpg', 'ALEI8.jpg', 'ALIONMASK.jpg', 'ALOCK6.jpg', 'AMICROS25.jpg']
batch 3
files: ['AMICROS34.jpg', 'AMICROS40.jpg', 'APACKME24.jpg', 'APALMTRE3.jpg', 'AREDROCK.jpg', 'ASALTPEPP.jpg', 'ASTOOL33.jpg', 'ASTOOL42.jpg', 'ASTRHATFL.jpg', 'ATABLE8.jpg']
batch 4
files: ['ATRUNK36.jpg', 'AYELLROCK.jpg', 'Aballofyarn3.jpg', 'Abird21.jpg', 'Aclock213.jpg', 'Alei22.jpg', 'Amicroscope23.jpg', 'Ascale120.jpg', 'Aseashe34.jpg', 'Astuffra9.jpg']
batch 5
files: ['Atent8.jpg', 'B